# Detector de Objetos com PyTorch + YOLO11 + ONNX

**Objetivo:** usar um modelo YOLO11n (PyTorch), pré-treinado no COCO, para detectar objetos com bounding boxes, exibir as 80 classes em português e exportar o detector para ONNX.

**Fluxo:**
`Imagem → YOLO11 / PyTorch → Detecção → Classe (PT) + Confiança + Bounding Box → Exportação ONNX`

> A tradução dos nomes das classes não altera o que o modelo aprendeu. O modelo continua reconhecendo os mesmos 80 IDs de classe do COCO — aqui apenas mapeamos cada ID para o nome em português na hora de exibir o resultado.


## 1. Instalação das dependências
Nada precisa ser baixado manualmente aqui: os pacotes são instalados via `pip` e os pesos do YOLO11n são baixados automaticamente pela biblioteca `ultralytics` na primeira execução.

In [ ]:
!pip install -q ultralytics onnx onnxruntime


## 2. Imports

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("Torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())


## 3. Carregar o modelo YOLO11n pré-treinado (COCO)
Ao instanciar `YOLO("yolo11n.pt")`, os pesos são baixados automaticamente (arquivo pequeno, ~5-6 MB) direto dos servidores da Ultralytics. Não é preciso baixar nada manualmente antes.

In [ ]:
model = YOLO("yolo11n.pt")  # baixa os pesos automaticamente na 1ª execução
model.info()


## 4. Dicionário de tradução das 80 classes COCO (EN → PT)
Os IDs seguem exatamente a ordem usada internamente pelo YOLO/COCO (0 a 79).

In [ ]:
classes_coco80_pt = {
    0: "pessoa", 1: "bicicleta", 2: "carro", 3: "moto", 4: "avião",
    5: "ônibus", 6: "trem", 7: "caminhão", 8: "barco", 9: "semáforo",
    10: "hidrante", 11: "placa de pare", 12: "parquímetro", 13: "banco (praça)", 14: "pássaro",
    15: "gato", 16: "cachorro", 17: "cavalo", 18: "ovelha", 19: "vaca",
    20: "elefante", 21: "urso", 22: "zebra", 23: "girafa", 24: "mochila",
    25: "guarda-chuva", 26: "bolsa", 27: "gravata", 28: "mala", 29: "frisbee",
    30: "esquis", 31: "snowboard", 32: "bola esportiva", 33: "pipa", 34: "taco de beisebol",
    35: "luva de beisebol", 36: "skate", 37: "prancha de surf", 38: "raquete de tênis", 39: "garrafa",
    40: "taça de vinho", 41: "xícara", 42: "garfo", 43: "faca", 44: "colher",
    45: "tigela", 46: "banana", 47: "maçã", 48: "sanduíche", 49: "laranja",
    50: "brócolis", 51: "cenoura", 52: "cachorro-quente", 53: "pizza", 54: "rosquinha",
    55: "bolo", 56: "cadeira", 57: "sofá", 58: "planta em vaso", 59: "cama",
    60: "mesa de jantar", 61: "vaso sanitário", 62: "televisão", 63: "notebook (laptop)", 64: "mouse",
    65: "controle remoto", 66: "teclado", 67: "celular", 68: "micro-ondas", 69: "forno",
    70: "torradeira", 71: "pia", 72: "geladeira", 73: "livro", 74: "relógio",
    75: "vaso (decorativo)", 76: "tesoura", 77: "urso de pelúcia", 78: "secador de cabelo", 79: "escova de dente",
}

assert len(classes_coco80_pt) == 80


## 5. Salvar `classes_coco80_pt.txt`
Esse arquivo é um dos dois insumos que o app de Inferência Universal de Imagens vai usar (junto com o `.onnx`) para mostrar o nome da classe em português a partir do ID retornado pelo modelo.

In [ ]:
with open("classes_coco80_pt.txt", "w", encoding="utf-8") as f:
    for idx in range(80):
        f.write(f"{idx};{classes_coco80_pt[idx]}\n")

print("Arquivo classes_coco80_pt.txt gerado.")


## 6. Função de inferência
Recebe o caminho de uma imagem, roda a detecção e devolve os resultados já com a classe traduzida.

In [ ]:
def detectar(caminho_imagem, conf_min=0.25):
    resultados = model.predict(source=caminho_imagem, conf=conf_min, verbose=False)
    r = resultados[0]

    deteccoes = []
    for box in r.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        deteccoes.append({
            "classe_id": cls_id,
            "classe_pt": classes_coco80_pt.get(cls_id, f"classe_{cls_id}"),
            "confianca": conf,
            "bbox": (x1, y1, x2, y2),
        })
    return r, deteccoes


## 7. Função para desenhar as bounding boxes com classe (PT) + confiança

In [ ]:
def desenhar_deteccoes(caminho_imagem, deteccoes):
    img = cv2.imread(caminho_imagem)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for d in deteccoes:
        x1, y1, x2, y2 = [int(v) for v in d["bbox"]]
        label = f'{d["classe_pt"]} {d["confianca"]:.2f}'
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, label, (x1, max(y1 - 8, 0)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 0, 0), 2, cv2.LINE_AA)

    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


## 8. Carregar uma imagem de teste
Você pode:
- fazer **upload de uma imagem sua** (célula abaixo, botão de upload do Colab), ou
- usar uma **URL de imagem de exemplo** (segunda opção, comentada).

Nenhuma dessas imagens precisa ser baixada com antecedência — é só rodar a célula.

In [ ]:
from google.colab import files

uploaded = files.upload()
caminho_imagem = list(uploaded.keys())[0]
print("Imagem carregada:", caminho_imagem)

# Alternativa por URL, se preferir não fazer upload manual:
# import urllib.request
# url = "https://ultralytics.com/images/bus.jpg"
# caminho_imagem = "teste.jpg"
# urllib.request.urlretrieve(url, caminho_imagem)


## 9. Rodar a detecção e exibir o resultado

In [ ]:
r, deteccoes = detectar(caminho_imagem)

for d in deteccoes:
    print(f'{d["classe_pt"]:<20} conf={d["confianca"]:.2f}  bbox={tuple(round(v,1) for v in d["bbox"])}')

desenhar_deteccoes(caminho_imagem, deteccoes)


## 10. Exportar o modelo para ONNX
O Ultralytics já cuida da exportação — o arquivo `yolo11n.onnx` é gerado na pasta local do Colab.

In [ ]:
caminho_onnx = model.export(format="onnx", imgsz=640, simplify=True)
print("Modelo exportado para:", caminho_onnx)


## 11. Validar o modelo ONNX com `onnxruntime`
Teste rápido para confirmar que o `.onnx` roda de forma independente do PyTorch — é esse arquivo que o app final vai carregar.

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(str(caminho_onnx), providers=["CPUExecutionProvider"])
entrada_nome = sess.get_inputs()[0].name
saida_nomes = [o.name for o in sess.get_outputs()]

print("Entrada:", sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print("Saídas:", saida_nomes)

# Inferência de teste com tensor aleatório só para checar se o grafo roda sem erro
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
saida = sess.run(saida_nomes, {entrada_nome: dummy})
print("Execução ONNX ok. Shape da saída:", saida[0].shape)


## 12. Baixar os arquivos finais
Estes dois arquivos são os únicos insumos que o app **Inferência Universal de Imagens** precisa:
- `yolo11n.onnx` — o modelo para rodar a inferência;
- `classes_coco80_pt.txt` — a tabela ID → nome da classe em português.

In [ ]:
from google.colab import files as colab_files

colab_files.download(str(caminho_onnx))
colab_files.download("classes_coco80_pt.txt")


## Observações finais

- O modelo continua reconhecendo exatamente as **80 categorias do COCO**; a tradução só muda como o nome é exibido.
- Este projeto pode servir de base para um **detector especializado** (ex.: `capacete | sem capacete | colete | luva | celular`), mas isso exige treinar o modelo do zero com um dataset próprio anotado com essas classes — não é mais transfer learning direto sobre os pesos do COCO.
